In [285]:
import pandas as pd
import numpy as np

In [286]:
df_deliveries = pd.read_csv('CSV/deliveries.csv')
df_matches = pd.read_csv('CSV/matches.csv')

In [287]:
teams = [
    'Sunrisers Hyderabad', 'Mumbai Indians', 'Royal Challengers Bengaluru',
    'Kolkata Knight Riders', 'Punjab Kings', 'Chennai Super Kings',
    'Rajasthan Royals', 'Delhi Capitals', 'Gujarat Titans', 'Lucknow Super Giants'
]

In [288]:
name_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru'
}

for col in ['team1', 'team2', 'winner']:
    df_matches[col] = df_matches[col].replace(name_map)


for col in ['batting_team', 'bowling_team']:
    df_deliveries[col] = df_deliveries[col].replace(name_map)


In [289]:
df_matches = df_matches[df_matches['team1'].isin(teams)]
df_matches = df_matches[df_matches['team2'].isin(teams)]

df_deliveries = df_deliveries[df_deliveries['batting_team'].isin(teams)]
df_deliveries = df_deliveries[df_deliveries['bowling_team'].isin(teams)]


In [290]:
df_matches = df_matches[df_matches['method'].isna()]

In [291]:
df_matches['venue_clean'] = (
    df_matches['venue']
    .str.strip()
    .str.lower()
    .str.split(",").str[0]
)

In [292]:
venue_mapping = {
    # Delhi
    "feroz shah kotla": "arun jaitley stadium",
    "arun jaitley stadium": "arun jaitley stadium",

    # Ahmedabad
    "sardar patel stadium": "narendra modi stadium",
    "motera": "narendra modi stadium",
    "narendra modi stadium": "narendra modi stadium",

    # Mohali
    "punjab cricket association stadium": "punjab cricket association is bindra stadium",
    "punjab cricket association is bindra stadium": "punjab cricket association is bindra stadium",

    # Pune
    "subrata roy sahara stadium": "maharashtra cricket association stadium",
    "maharashtra cricket association stadium": "maharashtra cricket association stadium",

    # Abu Dhabi
    "sheikh zayed stadium": "zayed cricket stadium",
    "zayed cricket stadium": "zayed cricket stadium",

    # Chennai
    "ma chidambaram stadium": "ma chidambaram stadium",

    # Mumbai
    "wankhede stadium": "wankhede stadium",
    "brabourne stadium": "brabourne stadium",
    "dr dy patil sports academy": "dr dy patil sports academy",

    # Hyderabad
    "rajiv gandhi international stadium": "rajiv gandhi international stadium",

    # Visakhapatnam
    "dr. y.s. rajasekhara reddy aca-vdca cricket stadium": "dr. y.s. rajasekhara reddy aca-vdca cricket stadium",

    # Jaipur
    "sawai mansingh stadium": "sawai mansingh stadium",

    # Nagpur
    "vidarbha cricket association stadium": "vidarbha cricket association stadium",

    # Lucknow
    "bharat ratna shri atal bihari vajpayee ekana cricket stadium": "bharat ratna shri atal bihari vajpayee ekana cricket stadium",

    # Ranchi
    "jsca international stadium complex": "jsca international stadium complex",

    # Raipur
    "shaheed veer narayan singh international stadium": "shaheed veer narayan singh international stadium",

    # Indore
    "holkar cricket stadium": "holkar cricket stadium",

    # Bangalore
    "m chinnaswamy stadium": "m chinnaswamy stadium",

    # Kolkata
    "eden gardens": "eden gardens",

    # International (South Africa / UAE)
    "newlands": "newlands",
    "kingsmead": "kingsmead",
    "supersport park": "supersport park",
    "st george's park": "st george's park",
    "new wanderers stadium": "new wanderers stadium",
    "buffalo park": "buffalo park",
    "outsurance oval": "outsurance oval",
    "de beers diamond oval": "de beers diamond oval",
    "barsapara cricket stadium": "barsapara cricket stadium",
    "sharjah cricket stadium": "sharjah cricket stadium",
    "dubai international cricket stadium": "dubai international cricket stadium",
    "maharaja yadavindra singh international cricket stadium": "maharaja yadavindra singh international cricket stadium",
}


In [293]:
df_matches['venue_clean'] = (
    df_matches['venue_clean']
    .str.strip().str.lower()
    .replace(venue_mapping)
)


In [294]:
df_matches.drop('venue', axis=1,inplace=True)
df_matches.rename(columns={'venue_clean':'venue'},inplace=True)

In [295]:
df_matches = df_matches[['id','venue','winner','target_runs']]

In [296]:
df_matches.rename(columns={'id' : 'match_id'},inplace=True)

In [297]:
print(df_matches['match_id'].nunique())
print(df_deliveries['match_id'].nunique())

963
980


In [298]:
delivery = df_matches.merge(df_deliveries, on='match_id')

In [299]:
delivery = delivery[delivery['inning'] == 2]

In [300]:
delivery['current_score'] = delivery.groupby('match_id')['total_runs'].cumsum()

In [301]:
delivery['runs_left'] =  delivery['target_runs'] - delivery['current_score']

In [302]:
delivery['balls_left'] =  120 - (delivery['over'] * 6 + delivery['ball'])

In [303]:
wickets = delivery.groupby('match_id')['is_wicket'].cumsum()
delivery['wickets_left'] = 10 - wickets

In [304]:
delivery['crr'] = delivery['current_score'] * 6 / (120- delivery['balls_left'])

In [305]:
delivery['rrr'] = delivery['runs_left'] * 6 / delivery['balls_left']

In [306]:
def result(row):
    return 1 if row['batting_team'] == row['winner'] else 0

In [307]:
delivery['result'] = delivery.apply(result,axis=1)

In [308]:
valid_balls = delivery[delivery['extras_type'].isna() | 
                       delivery['extras_type'].isin(['legbyes', 'byes', 'noballs'])]

balls_faced = valid_balls.groupby('batter').size()

In [309]:
batter_stats = valid_balls.groupby('batter').agg(
    runs_scored=('batsman_runs', 'sum')
)
batter_stats['balls_faced'] = balls_faced
batter_stats['strike_rate'] = batter_stats['runs_scored'] / batter_stats['balls_faced'] * 100


In [310]:
delivery = delivery.merge(
    batter_stats.add_prefix('striker_'),
    left_on='batter',
    right_index=True,
    how='left'
)

delivery = delivery.merge(
    batter_stats.add_prefix('non_striker_'),
    left_on='non_striker',
    right_index=True,
    how='left'
)

In [311]:
delivery['match_id'].value_counts()

match_id
1359480    136
829811     134
829737     134
1426268    133
1426288    132
          ... 
1304082     49
1082626     49
336021      38
1178424     20
829813       7
Name: count, Length: 961, dtype: int64

In [312]:
last_5_matches = (
    delivery.groupby('batter')['match_id'].apply(lambda x: x.drop_duplicates().nlargest(5))
)

In [ ]:
per_match = (
    delivery.groupby(['batter', 'match_id']).agg(
        runs_in_match=('batsman_runs', 'sum'),
        balls_in_match=('ball', 'count')
    )
    .reset_index()
)


per_match = per_match.sort_values(['batter', 'match_id'], ascending=[True, False])

last5 = (
    per_match.groupby('batter')
    .head(5)
    .groupby('batter')
    .agg(
        last5_runs=('runs_in_match', lambda x: list(x)),
        last5_balls=('balls_in_match', lambda x: list(x))
    )
)



In [333]:
last5['last5_runs_scored'] = last5['last5_runs'].apply(sum)
last5['last5_balls_faced'] = last5['last5_balls'].apply(sum)
last5['last5_SR'] = last5['last5_runs_scored'] / last5['last5_balls_faced'] * 100

In [337]:
last5.sort_values(by='last5_SR', ascending=False)

,last5_runs,last5_balls,last5_runs_scored,last5_balls_faced,last5_SR
batter,,,,,
L Wood,[9],[3],9,3,300.000000
OF Smith,[25],[9],25,9,277.777778
B Stanlake,[5],[2],5,2,250.000000
Shahid Afridi,"[1, 0, 33, 20]","[2, 1, 14, 6]",54,23,234.782609
Abhishek Sharma,"[66, 75, 15, 31, 37]","[30, 30, 9, 15, 12]",224,96,233.333333
...,...,...,...,...,...
KM Asif,[0],[2],0,2,0.000000
BAW Mendis,[0],[1],0,1,0.000000
KR Sen,[0],[3],0,3,0.000000


In [315]:
final_df = delivery[['match_id','batting_team','bowling_team','venue','runs_left','balls_left','wickets_left','target_runs','crr','rrr','result']]

In [316]:
final_df = final_df[final_df['balls_left'] != 0]

In [317]:
final_df.dropna(subset=['rrr'],inplace=True)

In [318]:
final_df.columns

Index(['match_id', 'batting_team', 'bowling_team', 'venue', 'runs_left',
       'balls_left', 'wickets_left', 'target_runs', 'crr', 'rrr', 'result'],
      dtype='object')

In [319]:
print(final_df.head(1))

     match_id                 batting_team           bowling_team  \
124    335982  Royal Challengers Bengaluru  Kolkata Knight Riders   

                     venue  runs_left  balls_left  wickets_left  target_runs  \
124  m chinnaswamy stadium      222.0         119            10        223.0   

     crr        rrr  result  
124  6.0  11.193277       0  


In [320]:
def phase_and_progress(row, total_overs=20):
    balls_bowled = total_overs*6 - row['balls_left']
    
    if row['balls_left'] <= 30:
        phase = 'death_over'
    elif row['balls_left'] >= 84:
        phase = 'powerplay'
    else:
        phase = 'middle_over'
    
    overs_completed = balls_bowled / 6
    
    return pd.Series([phase, overs_completed], index=['phase', 'overs_completed'])

In [321]:
final_df[['current_phase','over_completed']] = final_df.apply(phase_and_progress, axis=1)

In [322]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [323]:
unique_matches = final_df['match_id'].unique()
train_matches, test_matches = train_test_split(
    unique_matches, test_size=0.2, random_state=2
)

In [324]:
train_data = final_df[final_df['match_id'].isin(train_matches)]
test_data = final_df[final_df['match_id'].isin(test_matches)]

In [325]:
train_data['match_id'].value_counts().sort_values()

match_id
1178424     20
1082626     49
598068      50
1136608     51
1254093     53
          ... 
1426288    131
1426268    132
829811     133
829737     133
1359480    135
Name: count, Length: 768, dtype: int64

In [326]:
test_data['match_id'].value_counts().sort_values()

match_id
829813       7
336021      38
1304082     49
829803      63
1254087     65
          ... 
392190     129
1426294    130
1216522    130
829777     130
1370350    131
Name: count, Length: 193, dtype: int64

In [327]:
train_data = train_data.sample(frac = 1, random_state=42).reset_index(drop=True)
test_data = test_data.sample(frac = 1, random_state=42).reset_index(drop=True)

In [328]:
X_train = train_data.drop(columns=['result','match_id'])
y_train = train_data['result']

X_test = test_data.drop(columns=['result','match_id'])
y_test = test_data['result']

In [329]:
trf = ColumnTransformer(
    transformers=[
        ('trf', OneHotEncoder(sparse_output=False, drop='first'), ['batting_team', 'bowling_team', 'venue'])
    ], remainder='passthrough'
)

In [330]:
pipe = Pipeline(steps=[
    ('step1',trf),
    ('step2',LogisticRegression(solver='liblinear'))
])